# AIGC 5005 · Module 01 · Part 4
## Editor and notebook workflow

Companion notebook to Part 4 of the Module 01 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 20 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module01-tool` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module01-tool python=3.11 ipykernel`, then `conda activate module01-tool`. If VS Code offers to install `ipykernel`, accept.

## 1. Which interpreter is this notebook using?

A notebook needs the `ipykernel` package in the environment it runs in. VS Code offers to install it when you pick a kernel. The cell below shows which environment this notebook is running in and which version of `ipykernel` it found. If an installed package ever seems to be "missing", compare this interpreter with the one you installed into.

In [ ]:
import sys

import ipykernel

print("Interpreter     :", sys.executable)
print("ipykernel       :", ipykernel.__version__)

For a project that has notebooks, list `ipykernel` in `requirements.txt`, pinned, like any other package.

## 2. Hidden state

A notebook keeps the results of every cell you have run, **including cells you have since edited or deleted**. That makes a notebook feel like it works, when run top to bottom in a fresh session it may not.

Below, a tiny helper plays the part of the notebook's kernel. `Session` remembers variables between cells, exactly as the real kernel does, and `fresh_run` runs a list of cells in a brand-new session, which is what *Restart and Run All* does.

In [ ]:
class Session:
    """A stand-in for a notebook kernel: it remembers variables between cells."""

    def __init__(self):
        self.variables = {}

    def run(self, cell):
        exec(cell, self.variables)


def fresh_run(cells):
    """Run the cells in order in a brand-new session, as Restart and Run All would."""
    session = Session()
    for number, cell in enumerate(cells, start=1):
        try:
            session.run(cell)
        except Exception as error:
            print(f"  cell {number} failed: {type(error).__name__}: {error}")
            return False
    print("  all cells ran")
    return True


print("Helpers ready.")

### The story

You write a notebook with three cells. While working, you run an extra cell that defines `price`, then **delete that cell** because it looks like a stray experiment.

**Predict first:** does the remaining notebook still work in your session? Will it work for someone else?

In [ ]:
kernel = Session()

kernel.run("price = 100")                  # the cell you later delete
notebook_cells = [
    "tax_rate = 0.13",
    "total = price * (1 + tax_rate)",       # quietly depends on the deleted cell
    "print('Total:', round(total, 2))",
]

print("In your session, running the remaining cells one at a time:")
for cell in notebook_cells:
    kernel.run(cell)                        # works: the kernel still remembers price

It works, because the kernel still remembers `price`. Now run the same three cells the way a marker or a classmate would, in a fresh session.

In [ ]:
print("In a fresh session:")
fresh_run(notebook_cells)

`NameError: name 'price' is not defined`. The notebook was never correct; your session was hiding the problem.

### The fix

Before you submit or share a notebook: **restart the kernel and run all cells** (in VS Code, *Restart* then *Run All* in the notebook toolbar). Fix whatever fails. Here the repair is to put the definition back into a visible cell, in the right order.

In [ ]:
repaired = [
    "price = 100",
    "tax_rate = 0.13",
    "total = price * (1 + tax_rate)",
    "print('Total:', round(total, 2))",
]
print("Repaired notebook, fresh session:")
fresh_run(repaired)

## 3. Order matters too

Running cells out of order has the same effect in the other direction: a cell can work today because you ran a later cell first.

In [ ]:
out_of_order = [
    "print(discount)",                       # uses discount before it is defined
    "discount = 0.1",
]
print("Run top to bottom in a fresh session:")
fresh_run(out_of_order)

## 4. Script or notebook?

| Use a script when | Use a notebook when |
|---|---|
| The code needs to run reliably start to finish | You are exploring and want to run pieces |
| Someone else will run it | You are explaining a result with commentary |
| It will be tested or deployed | You are inspecting data or plotting |
| It is part of a package | The work is throwaway |

For this module, work in a script; the labs say which format is expected.

---
## Try it

Take any notebook from this course, change a cell you have already run (for example, rename a variable), then use *Restart* and *Run All*. If something fails, you have found hidden state. Fix it, and make a habit of doing this before you submit.